In [5]:
import pandas as pd
import numpy as np
import math
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [6]:
train = pd.read_csv("../data/processed/train.csv")
test = pd.read_csv("../data/processed/test.csv")
print("Train:", train.shape)
display(train.head())
print("Test:", test.shape)
display(test.head())

Train: (88799, 6)


,reviewers,course_id,rating,date_reviews,user_id,course_idx
0,By George W,philosophy,5,2015-08-07,2736,384
1,By Konstantinos K,learning-how-to-learn,5,2015-08-07,4049,314
2,By Mohammed R,machine-learning,5,2015-08-07,5116,318
3,By Michelle F,learning-how-to-learn,5,2015-08-07,4948,314
4,By Tushar T,machine-learning,5,2015-08-07,7579,318


Test: (16342, 6)


,reviewers,course_id,rating,date_reviews,user_id,course_idx
0,By Manojkumar P,wharton-customer-analytics,4,2015-12-29,4472,542
1,By Julien H J,wharton-customer-analytics,5,2016-01-13,3783,542
2,By Jingxia P,wharton-operations-analytics,3,2016-01-14,3486,551
3,By Roussel K,python,5,2016-01-14,6470,420
4,By Manojkumar P,terrorism,4,2016-01-16,4472,501


In [7]:
print("Các cột train:", train.columns.tolist())
print("Các cột test:", test.columns.tolist())

print("\nThông tin tập train:")
train.info()

Các cột train: ['reviewers', 'course_id', 'rating', 'date_reviews', 'user_id', 'course_idx']
Các cột test: ['reviewers', 'course_id', 'rating', 'date_reviews', 'user_id', 'course_idx']

Thông tin tập train:
<class 'pandas.DataFrame'>
RangeIndex: 88799 entries, 0 to 88798
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   reviewers     88799 non-null  str  
 1   course_id     88799 non-null  str  
 2   rating        88799 non-null  int64
 3   date_reviews  88799 non-null  str  
 4   user_id       88799 non-null  int64
 5   course_idx    88799 non-null  int64
dtypes: int64(3), str(3)
memory usage: 7.6 MB


### Baseline 1 — Popularity-Based Recommendation
Đề xuất các khoá học phổ biến nhất dựa trên số lượt đánh giá tích cực trong train

In [8]:
nguong_tich_cuc = 4
positive_train = train[
    train["rating"] >= nguong_tich_cuc
].copy()
mon_pho_bien = (
    positive_train.groupby("course_idx")["course_idx"]
    .count()
    .reset_index(name= "đánh giá tích cực")
    .sort_values("đánh giá tích cực", ascending= False)
)
print("Số lượt đánh giá tích cực:", len(positive_train))
print("Số khoá học có ít nhất 1 đánh giá tích cực:", len(mon_pho_bien))
display(mon_pho_bien.head(10))

Số lượt đánh giá tích cực: 81969
Số khoá học có ít nhất 1 đánh giá tích cực: 564


,course_idx,đánh giá tích cực
362,362,2086
318,318,2073
420,420,1995
314,314,1893
423,423,1820
498,498,1398
121,121,1354
431,431,1297
557,557,1268
321,321,1075


In [ ]:
# Các khóa học mà mỗi người dùng đã tương tác trong train
khoa_hoc_da_hoc = (
    train.groupby("user_id")["course_idx"]
    .apply(set)
    .to_dict()
)

# Danh sách khóa học sắp xếp theo độ phổ biến
danh_sach_pho_bien = mon_pho_bien["course_idx"].tolist()

def goi_y_top_n(user_id, n=10):
    da_tuong_tac = khoa_hoc_da_hoc.get(user_id, set())

    goi_y = [
        course_idx
        for course_idx in danh_sach_pho_bien
        if course_idx not in da_tuong_tac
    ]

    return goi_y[:n]

# Thử đề xuất cho một người dùng trong train
user_thu = train["user_id"].iloc[1]

print("User ID:", user_thu)
print("Các khóa học đã tương tác:", len(
    khoa_hoc_da_hoc.get(user_thu, set())
))
print("Top-10 đề xuất:", goi_y_top_n(user_thu, 10))

User ID: 4049
Các khóa học đã tương tác: 9
Top-10 đề xuất: [318, 420, 423, 498, 431, 557, 321, 207, 116, 174]


In [16]:
# Tập khóa học liên quan trong test của mỗi người dùng
test_tich_cuc = test[test["rating"] >= nguong_tich_cuc]

tap_lien_quan = (
    test_tich_cuc.groupby("user_id")["course_idx"]
    .apply(set)
    .to_dict()
)

so_user_test = test["user_id"].nunique()
so_user_co_lien_quan = len(tap_lien_quan)
so_user_khong_co_lien_quan = so_user_test - so_user_co_lien_quan

print("Số user trong test:", so_user_test)
print("Số user có ít nhất 1 khóa học tích cực:", so_user_co_lien_quan)
print("Số user không có khóa học tích cực:", so_user_khong_co_lien_quan)

Số user trong test: 8218
Số user có ít nhất 1 khóa học tích cực: 8101
Số user không có khóa học tích cực: 117


In [ ]:
K = 10
def tinh_precision(recommended, relevant, k=10):
    top_k = recommended[:k]
    return len(set(top_k) & relevant) / k
def tinh_recall(recommended, relevant, k=10):
    top_k = recommended[:k]
    return len(set(top_k) & relevant) / len(relevant)
def tinh_f1(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)
def tinh_ndcg(recommended, relevant, k=10):
    top_k = recommended[:k]

    dcg = sum(
        1 / math.log2(rank + 2)
        for rank, course in enumerate(top_k)
        if course in relevant
    )

    ideal_length = min(len(relevant), k)

    idcg = sum(
        1 / math.log2(rank + 2)
        for rank in range(ideal_length)
    )

    return dcg / idcg if idcg > 0 else 0.0
# Đánh giá trên những user có ít nhất một khóa học tích cực trong test
ket_qua = []
for user_id, relevant in tap_lien_quan.items():
    recommended = goi_y_top_n(user_id, K)
    precision = tinh_precision(recommended, relevant, K)
    recall = tinh_recall(recommended, relevant, K)
    f1 = tinh_f1(precision, recall)
    ndcg = tinh_ndcg(recommended, relevant, K)
    ket_qua.append({
        "user_id": user_id,
        "Precision@10": precision,
        "Recall@10": recall,
        "F1@10": f1,
        "NDCG@10": ndcg
    })
ket_qua_df = pd.DataFrame(ket_qua)
print("Số user được đánh giá:", len(ket_qua_df))
print("\nKết quả Popularity Baseline:")
display(
    ket_qua_df[
        ["Precision@10", "Recall@10", "F1@10", "NDCG@10"]
    ].mean().to_frame("Giá trị trung bình")
)

Số user được đánh giá: 8101

Kết quả Popularity Baseline:


,Giá trị trung bình
Precision@10,0.034650
Recall@10,0.187014
F1@10,0.058167
NDCG@10,0.114803


### Baseline 2 - Rating Prediction
+ Global Mean: dự đoán bằng điểm đánh giá trung bình của toàn bộ tập train.
+ Item Average: dự đoán bằng điểm đánh giá trung bình của từng khóa học trong tập train.

#### Global Mean

In [12]:
global_mean = train["rating"].mean()
global_mean

np.float64(4.62680886046014)

In [13]:
# Rating thực tế trong tập test
y_true = test["rating"].to_numpy()

# Dự đoán bằng trung bình rating của train
y_pred_global = np.full(
    shape=len(test),
    fill_value=global_mean
)

# Đánh giá
rmse_global = np.sqrt(
    mean_squared_error(y_true, y_pred_global)
)

mae_global = mean_absolute_error(y_true, y_pred_global)

print(f"Global Mean: {global_mean:.4f}")
print(f"RMSE: {rmse_global:.4f}")
print(f"MAE:  {mae_global:.4f}")

Global Mean: 4.6268
RMSE: 0.7862
MAE:  0.5561


#### Item Average 

In [14]:
# Tính rating trung bình của từng khóa học trên train
diem_trung_binh_khoa_hoc = (
    train.groupby("course_idx")["rating"]
    .mean()
)

# Dự đoán rating cho từng tương tác trong test
y_pred_item = (
    test["course_idx"]
    .map(diem_trung_binh_khoa_hoc)
    .fillna(global_mean)
    .to_numpy()
)

# Rating thực tế trong test
y_true = test["rating"].to_numpy()

# Tính RMSE và MAE
rmse_item = np.sqrt(
    mean_squared_error(y_true, y_pred_item)
)

mae_item = mean_absolute_error(y_true, y_pred_item)

print(f"RMSE: {rmse_item:.4f}")
print(f"MAE:  {mae_item:.4f}")

RMSE: 0.7525
MAE:  0.4977
